# 02 — Preprocessing Validation

Validates the Python pipeline against MATLAB output before Phase 2.
Run a subject through both pipelines, then compare overlaid.

**What to check:**
- Filtered marker trajectories match MATLAB `filterMarkerData.m` to within numerical precision
- GRF filtering matches `filtForceCOP.m` (forces at 8 Hz, COP at 15 Hz, moments unfiltered)
- Heel strike / toe-off indices match `gaitEventDetection.m` (threshold=15 N, min peak=150 N)
- Time-normalized waveforms match `gaitCycleNormalization.m` (101-point pchip)
- Sagittal joint angles (knee, hip, ankle) are in the expected range and direction

**MATLAB export instructions** (Section 6):  
Run `export_for_python_validation.m` (or equivalent) to write filtered marker data,
filtered GRF, heel-strike indices, and time-normalized joint angles to CSV.
Place the exports in `data/matlab_validation/{SUBJECT}/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from gait_ml.config import DEFAULT_LAB_CONFIG as cfg
from gait_ml.io import load_force_tsv, load_marker_tsv, load_subject_weight_newtons
from gait_ml.kinematics import ankle_dorsiflexion, hip_flexion, knee_flexion
from gait_ml.preprocessing import (
    normalize_all_cycles,
    preprocess_forces,
    preprocess_markers,
)
from gait_ml.segmentation import detect_gait_events_grf

RAW = Path('../data/raw')
MATLAB_DIR = Path('../data/matlab_validation')

SUBJECT = 'FS6'
CONDITION = 'WalkingPreferred'
TRIAL = 1

FS_KIN = cfg.acquisition.kinematic_sample_rate_hz   # 160 Hz
FS_GRF = cfg.acquisition.grf_sample_rate_hz         # 1120 Hz
GRF_KIN_RATIO = int(cfg.acquisition.grf_kinematic_ratio)  # 7

stem = f'{SUBJECT}_{CONDITION}{TRIAL}'
print(f'Trial: {stem}')

## 1. Load and preprocess markers

In [ ]:
markers_raw = load_marker_tsv(RAW / f'{stem}.tsv')
markers_proc = preprocess_markers(markers_raw)

# Build marker_names list and data array for kinematics functions.
# Columns are CLAVX CLAVY CLAVZ C7X ... (every 3 = one marker).
marker_cols = [c for c in markers_proc.columns if c not in ('Frame', 'Time')]
marker_names = [c[:-1] for c in marker_cols[::3]]   # strip trailing X
data = markers_proc[marker_cols].to_numpy()

t_kin = markers_proc['Time'].to_numpy()

print(f'Frames: {len(markers_proc)}  ({len(markers_proc)/FS_KIN:.1f} s)')
print(f'Markers: {len(marker_names)}')
print(f'Remaining NaN after gap-fill: {np.isnan(data).sum()}')

In [ ]:
# Spot-check: LKNEE Y (vertical position) raw vs filtered
col_idx = marker_cols.index('LKNEEY')

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t_kin, markers_raw[marker_cols].to_numpy()[:, col_idx], lw=0.8, alpha=0.5, label='raw')
ax.plot(t_kin, data[:, col_idx], lw=1.2, label='filtered (8 Hz)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('mm')
ax.set_title('LKNEE Y — raw vs filtered')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Load and preprocess GRF

In [ ]:
force_l_raw = load_force_tsv(RAW / f'{stem}_f_4.tsv')
force_r_raw = load_force_tsv(RAW / f'{stem}_f_5.tsv')
force_l = preprocess_forces(force_l_raw)
force_r = preprocess_forces(force_r_raw)

t_grf = force_l['TIME'].to_numpy()
fz_l = force_l['Force_Z'].to_numpy()
fz_r = force_r['Force_Z'].to_numpy()

# Verify moments are NOT filtered (raw == processed)
moment_cols = list(cfg.qualisys.moment_columns)
for col in moment_cols:
    if col in force_l.columns:
        assert np.allclose(
            force_l_raw[col].dropna(), force_l[col].dropna()
        ), f'{col} should be unfiltered'
print('Moment columns unfiltered: OK')

print(f'GRF frames: {len(force_l)}  ratio to kin: {len(force_l)/len(markers_proc):.1f}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t_grf, fz_l, lw=0.8, label='Left belt')
ax.plot(t_grf, fz_r, lw=0.8, label='Right belt')
ax.axhline(cfg.gait_events.threshold_n, color='k', ls='--', lw=0.8,
           label=f'Threshold ({cfg.gait_events.threshold_n:.0f} N)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Fz (N)')
ax.set_title('Vertical GRF — both belts (filtered)')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Gait event detection

In [ ]:
events_l = detect_gait_events_grf(fz_l, FS_GRF)
events_r = detect_gait_events_grf(fz_r, FS_GRF)

print(f'Left:  {len(events_l["heel_strike"])} heel strikes, {len(events_l["toe_off"])} toe-offs')
print(f'Right: {len(events_r["heel_strike"])} heel strikes, {len(events_r["toe_off"])} toe-offs')

# Cycle durations
for side, ev in [('L', events_l), ('R', events_r)]:
    hs = ev['heel_strike']
    if len(hs) > 1:
        durations_s = np.diff(hs) / FS_GRF
        print(f'  {side} cycle duration: {durations_s.mean():.3f} ± {durations_s.std():.3f} s '
              f'(expected ~1.0–1.3 s walking)')

In [ ]:
# Plot first 10 s with events marked
mask = t_grf < 10.0

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
for ax, fz, ev, label in zip(
    axes,
    [fz_l, fz_r],
    [events_l, events_r],
    ['Left belt', 'Right belt'],
):
    ax.plot(t_grf[mask], fz[mask], lw=0.9)
    hs_t = t_grf[ev['heel_strike']]
    to_t = t_grf[ev['toe_off']]
    ax.vlines(hs_t[hs_t < 10], 0, fz.max(), color='C2', lw=0.8, label='heel strike')
    ax.vlines(to_t[to_t < 10], 0, fz.max(), color='C3', lw=0.8, ls='--', label='toe-off')
    ax.set_ylabel('Fz (N)')
    ax.set_title(label)
    ax.legend(fontsize=8)

axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

## 4. Time normalization

Each gait cycle → 101 points (0–100% via pchip), matching `gaitCycleNormalization.m`.

In [ ]:
# Time-normalize vertical GRF cycles (left belt)
fz_norm_l = normalize_all_cycles(fz_l, events_l['heel_strike'])   # (n_cycles, 101)
fz_norm_r = normalize_all_cycles(fz_r, events_r['heel_strike'])

bw_n = load_subject_weight_newtons(SUBJECT, RAW)
pct = np.arange(101)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, fz_norm, label in zip(axes, [fz_norm_l, fz_norm_r], ['Left', 'Right']):
    mean = fz_norm.mean(axis=0) / bw_n
    sd = fz_norm.std(axis=0) / bw_n
    ax.plot(pct, mean, lw=1.5)
    ax.fill_between(pct, mean - sd, mean + sd, alpha=0.25)
    ax.axhline(1.0, color='k', ls='--', lw=0.7, label='1 BW')
    ax.set_xlabel('Gait cycle (%)')
    ax.set_ylabel('Fz (BW)')
    ax.set_title(f'{label} belt — {len(fz_norm)} cycles')
    ax.legend(fontsize=8)

fig.suptitle(f'{SUBJECT} {CONDITION} — normalized Fz (mean ± SD)', y=1.01)
plt.tight_layout()
plt.show()

print(f'Left  peak Fz: {fz_norm_l.max(axis=1).mean()/bw_n:.2f} ± {fz_norm_l.max(axis=1).std()/bw_n:.3f} BW')
print(f'Right peak Fz: {fz_norm_r.max(axis=1).mean()/bw_n:.2f} ± {fz_norm_r.max(axis=1).std()/bw_n:.3f} BW')
print('Expected walking: ~1.1–1.3 BW with M-shape double bump')

## 5. Sagittal joint angles

GRF heel strikes are converted to kinematic frame indices by dividing
by the GRF/kinematic ratio (7).

Knee angles use a **static reference correction**: the knee angle
computed from the Tcap (T-pose) standing trial is subtracted to correct
for the AP offset of the greater trochanter marker from the femoral axis.
This matches MATLAB's `refknee` correction.

In [ ]:
from gait_ml.kinematics import static_knee_reference

# Map GRF heel strike indices → kinematic frame indices
hs_kin_l = events_l['heel_strike'] // GRF_KIN_RATIO
hs_kin_r = events_r['heel_strike'] // GRF_KIN_RATIO

# Load Tcap static trial for reference angle correction
tcap_path = RAW / f'{SUBJECT}_Tcap1.tsv'
tcap_raw = load_marker_tsv(tcap_path)
tcap_proc = preprocess_markers(tcap_raw)
tcap_cols = [c for c in tcap_proc.columns if c not in ('Frame', 'Time')]
tcap_names = [c[:-1] for c in tcap_cols[::3]]
tcap_data = tcap_proc[tcap_cols].to_numpy()

ref_knee_l = static_knee_reference(tcap_data, tcap_names, side='L')
ref_knee_r = static_knee_reference(tcap_data, tcap_names, side='R')
print(f'Static knee reference — L: {ref_knee_l:.2f}°  R: {ref_knee_r:.2f}°')
print(f'MATLAB refknee        — L: 7.71°  R: 1.32°  (from results_kneeAngle_walking.csv)')
print()

# Compute joint angles with static correction applied to knee
knee_l = knee_flexion(data, marker_names, side='L', static_ref=ref_knee_l)
knee_r = knee_flexion(data, marker_names, side='R', static_ref=ref_knee_r)
hip_l  = hip_flexion(data, marker_names, side='L')
hip_r  = hip_flexion(data, marker_names, side='R')
ank_l  = ankle_dorsiflexion(data, marker_names, side='L')
ank_r  = ankle_dorsiflexion(data, marker_names, side='R')

print(f'Knee L range: {knee_l.min():.1f}° to {knee_l.max():.1f}°  (expect ~0–65° walking)')
print(f'Hip  L range: {hip_l.min():.1f}° to {hip_l.max():.1f}°')
print(f'Ankle L range: {ank_l.min():.1f}° to {ank_l.max():.1f}°')


In [ ]:
angles = {
    'Knee flex (L)': (knee_l, hs_kin_l),
    'Knee flex (R)': (knee_r, hs_kin_r),
    'Hip flex (L)':  (hip_l,  hs_kin_l),
    'Hip flex (R)':  (hip_r,  hs_kin_r),
    'Ankle df (L)':  (ank_l,  hs_kin_l),
    'Ankle df (R)':  (ank_r,  hs_kin_r),
}

fig, axes = plt.subplots(3, 2, figsize=(12, 10), sharex=True)
for ax, (label, (signal, hs)) in zip(axes.flat, angles.items()):
    if len(hs) < 2:
        ax.set_title(f'{label} — insufficient cycles')
        continue
    norm = normalize_all_cycles(signal, hs)   # (n_cycles, 101)
    mean = norm.mean(axis=0)
    sd = norm.std(axis=0)
    ax.plot(pct, mean, lw=1.5)
    ax.fill_between(pct, mean - sd, mean + sd, alpha=0.25)
    ax.axhline(0, color='k', lw=0.5)
    ax.set_ylabel('Degrees')
    ax.set_title(f'{label}  (n={len(norm)})')

for ax in axes[-1]:
    ax.set_xlabel('Gait cycle (%)')

fig.suptitle(
    f'{SUBJECT} {CONDITION} — sagittal joint angles (mean ± SD)\n'
    'NOTE: sign convention not yet validated against MATLAB',
    y=1.01,
)
plt.tight_layout()
plt.show()

## 6. MATLAB comparison

Uses MATLAB pipeline output already available on disk.

| File | Contents |
|------|----------|
| `results_kneeAngle_walking.csv` | Per-cycle scalars: `maxknee`, `msknee`, `refknee` (degrees) |
| `results_peakForces_walking.csv` | Per-stance peak Fz in N and normalized to BW, both belts |

**Comparisons run:**
- Peak knee flexion per cycle — `max(corrected_knee_cycle)` vs MATLAB `maxknee`
- Peak Fz per stance normalized to BW

**`msknee` is not compared** — MATLAB stores it as the raw (uncorrected)
minimum knee angle during stance. Evidence: `msknee ≈ refknee` for all
cycles (~7.71°). After static correction, the corrected midstance angle
≈ 0° by construction, making a comparison with raw MATLAB `msknee`
misleading. Peak flexion is the unambiguous check.

Pass/fail criteria:
- Peak knee flexion: mean difference < 3°, no cycle > 5° off
- Peak Fz (BW): mean difference < 0.05 BW

In [ ]:
import os, pathlib

# Set MATLAB_RESULTS_DIR env var to point at your local MATLAB output directory.
# This path is machine-specific and not committed to the repo.
_matlab_dir = os.environ.get('MATLAB_RESULTS_DIR', '')
if not _matlab_dir:
    raise EnvironmentError(
        'Set MATLAB_RESULTS_DIR to the directory containing results_kneeAngle_walking.csv '
        'and results_peakForces_walking.csv before running this cell.'
    )
MATLAB_RESULTS = pathlib.Path(_matlab_dir)

kw = pd.read_csv(MATLAB_RESULTS / 'results_kneeAngle_walking.csv')
pk = pd.read_csv(MATLAB_RESULTS / 'results_peakForces_walking.csv')

mask_kw = (kw['subID'] == SUBJECT) & (kw['condID'] == CONDITION) & (kw['trial'] == TRIAL)
matlab_knee_l = kw[mask_kw & (kw['side'] == 'L')]['maxknee'].to_numpy()
matlab_knee_r = kw[mask_kw & (kw['side'] == 'R')]['maxknee'].to_numpy()
matlab_ms_l   = kw[mask_kw & (kw['side'] == 'L')]['msknee'].to_numpy()

mask_pk = (pk['subject'] == SUBJECT) & (pk['condition'] == CONDITION) & (pk['trial'] == TRIAL)
matlab_peakbw_l = pk[mask_pk & (pk['belt'] == 'L')]['peakForceBW'].to_numpy()
matlab_peakbw_r = pk[mask_pk & (pk['belt'] == 'R')]['peakForceBW'].to_numpy()

print(f'MATLAB cycles  — knee L: {len(matlab_knee_l)}  knee R: {len(matlab_knee_r)}')
print(f'Python cycles  — knee L: {len(hs_kin_l)-1}  knee R: {len(hs_kin_r)-1}')
print(f'MATLAB stances — left belt: {len(matlab_peakbw_l)}  right belt: {len(matlab_peakbw_r)}')

In [ ]:
# ---- Peak knee flexion per cycle ----
knee_l_norm = normalize_all_cycles(knee_l, hs_kin_l)   # (n_cycles, 101)
knee_r_norm = normalize_all_cycles(knee_r, hs_kin_r)

py_maxknee_l = knee_l_norm.max(axis=1)
py_maxknee_r = knee_r_norm.max(axis=1)

n_l = min(len(py_maxknee_l), len(matlab_knee_l))
n_r = min(len(py_maxknee_r), len(matlab_knee_r))

print('=== Peak knee flexion (deg) ===')
print(f'  Python L:  {py_maxknee_l[:n_l].mean():.2f} +/- {py_maxknee_l[:n_l].std():.2f}')
print(f'  MATLAB L:  {matlab_knee_l[:n_l].mean():.2f} +/- {matlab_knee_l[:n_l].std():.2f}')
diff_pk = py_maxknee_l[:n_l] - matlab_knee_l[:n_l]
print(f'  Mean diff: {diff_pk.mean():.2f}  max abs diff: {np.abs(diff_pk).max():.2f}  (pass: mean<3, max<5)')
print()

# Note: MATLAB msknee is NOT compared here.
# msknee ≈ refknee (both ~7.71 for FS6 L), meaning MATLAB stores msknee as
# the raw (uncorrected) minimum knee angle during stance. After static
# correction, corrected midstance ≈ 0° by construction (most-extended
# position ≈ standing reference). The comparison would be corrected Python
# (~0°) vs uncorrected MATLAB (~7.75°) — apples to oranges.
# Peak flexion is the meaningful and unambiguous check from this CSV.
print('(msknee comparison omitted — see notebook markdown for explanation)')
print()

# ---- Peak Fz per stance ----
py_peakbw_l = np.array([
    fz_l[hs:to].max() / bw_n
    for hs, to in zip(events_l['heel_strike'], events_l['toe_off'])
])
py_peakbw_r = np.array([
    fz_r[hs:to].max() / bw_n
    for hs, to in zip(events_r['heel_strike'], events_r['toe_off'])
])

n_fl = min(len(py_peakbw_l), len(matlab_peakbw_l))
n_fr = min(len(py_peakbw_r), len(matlab_peakbw_r))

print('=== Peak Fz normalized to BW ===')
print(f'  Python L:  {py_peakbw_l[:n_fl].mean():.4f} +/- {py_peakbw_l[:n_fl].std():.4f}')
print(f'  MATLAB L:  {matlab_peakbw_l[:n_fl].mean():.4f} +/- {matlab_peakbw_l[:n_fl].std():.4f}')
diff_fz = py_peakbw_l[:n_fl] - matlab_peakbw_l[:n_fl]
print(f'  Mean diff: {diff_fz.mean():.4f} BW  (pass: <0.05 BW)')
print()

# ---- Scatter plots ----
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, py_vals, mat_vals, title in [
    (axes[0], py_maxknee_l[:n_l],  matlab_knee_l[:n_l],   'Peak knee flex L (deg)'),
    (axes[1], py_peakbw_l[:n_fl],  matlab_peakbw_l[:n_fl],'Peak Fz L (BW)'),
]:
    lo = min(py_vals.min(), mat_vals.min()) * 0.95
    hi = max(py_vals.max(), mat_vals.max()) * 1.05
    ax.scatter(mat_vals, py_vals, alpha=0.6, s=20)
    ax.plot([lo, hi], [lo, hi], 'k--', lw=0.8, label='identity')
    ax.set_xlabel('MATLAB')
    ax.set_ylabel('Python')
    ax.set_title(title)
    ax.legend(fontsize=8)

fig.suptitle(f'{SUBJECT} {CONDITION} trial {TRIAL} — Python vs MATLAB')
plt.tight_layout()
plt.show()


## 7. Validation summary

Fill in after running section 6.

| Check | Python mean ± SD | MATLAB mean ± SD | Mean diff | Pass? |
|-------|-----------------|-----------------|-----------|-------|
| Peak knee flex L (°) | — | — | — | — |
| Peak knee flex R (°) | — | — | — | — |
| Peak Fz L (BW) | — | — | — | — |
| Peak Fz R (BW) | — | — | — | — |

**Pass criteria:** peak knee mean diff < 3°; peak Fz mean diff < 0.05 BW

**Notes / discrepancies:**  
_(document any remaining offsets here)_

**Next steps once this passes:**
- Repeat for one running trial (belt selection already validated in notebook 01)
- Extend to hip and ankle once MATLAB output for those angles is available